Import Library

In [ ]:
# Import library 
import pandas as pd
import matplotlib.pyplot as plt
from kmodes.kmodes import KModes           

Load Data

In [ ]:
# load data
df_menu = pd.read_csv("data/menu_class.csv", encoding="latin1")
df_kmeans = pd.read_csv("data/kmeans.csv", encoding="latin1")

print("Menu class shape:", df_menu.shape)
print("KMeans dataset shape:", df_kmeans.shape)

In [ ]:
df_menu.rename(columns={"menu":"menu"}, inplace=True)
df_kmeans.rename(columns={"Menu":"menu"}, inplace=True)

# Pastikan huruf kecil semua (untuk mencegah mismatch)
df_menu["menu"] = df_menu["menu"].str.lower()
df_kmeans["menu"] = df_kmeans["menu"].str.lower()

In [ ]:
df_menu['Menu Item Class'].unique()

In [ ]:
print(df_menu.columns)
print(df_kmeans.columns)

In [ ]:
# merge
df_coba = df_menu.merge(df_kmeans, on="menu", how="left")

print("Final merged dataset shape:", df_coba.shape)

In [ ]:
df_coba['Menu Item Class'].unique()

In [ ]:
# select feature
cols_kmodes = ["Menu Item Class", "Cluster_Id"]

# Filter dataset sesuai kolom
df_kmodes = df_coba[cols_kmodes].copy()


In [ ]:
# handling missing value
df_kmodes = df_kmodes.dropna()
print("Setelah drop missing:", df_kmodes.shape)

In [ ]:
for col in df_kmodes.columns:
    df_kmodes[col] = df_kmodes[col].astype(str)

In [ ]:
df_kmodes

Clustering

In [ ]:
cost = []
range_n_clusters = [2, 3, 4, 5, 6, 7, 8]

for k in range_n_clusters:

    km = KModes(
        n_clusters=k,
        init='Huang',
        random_state=42,
        max_iter=50
    )

    km.fit_predict(df_kmodes)

    cost.append(km.cost_)

plt.figure(figsize=(8, 6))

plt.plot(
    range_n_clusters,
    cost,
    marker='o',
    linewidth=2
)

plt.xlabel("Number of Clusters (K)", fontsize=16)
plt.ylabel("K-Modes Cost", fontsize=16)

plt.xticks(range_n_clusters, fontsize=13)
plt.yticks(fontsize=13)

plt.tight_layout()

plt.show()

In [ ]:
km = KModes(
    n_clusters=3,
    init='Huang',
    random_state=42,
    max_iter=50
)

clusters = km.fit_predict(df_kmodes)
df_kmodes["cluster_kmodes"] = clusters

# summary
summary = df_kmodes.groupby(["cluster_kmodes", "Menu Item Class", "Cluster_Id"]).size().reset_index(name="count")
print("\nRingkasan cluster:")
print(summary)

In [ ]:
df_kmodes.to_csv('results.csv', index=False)

In [ ]:
summary.to_csv('summary.csv', index=False)